# CFPB Issue Taxonomy: audit, consolidation, and consumer-harm dimensions

**Goal.** Reduce the 33 raw CFPB `Issue` labels to a smaller set of interpretable
consumer-harm dimensions suitable for eventual lender x sub-product comparison.

**Two questions are kept separate throughout.**

1. *Conceptual* — is this complaint plausibly about lender or product conduct, or about the
   borrower's own circumstances? A category can be well differentiated across lenders and
   still be useless as a Safety Label signal.
2. *Empirical* — does the category actually separate payday lenders by more than sampling
   noise?

**Discipline.** No Safety Score, no severity weights, no clustering, no dimensionality
reduction, no interpretation of complaint counts as rates. Raw data is read-only. Nothing
is committed.

**Inputs:** `data/raw/paydayComplaints.csv`, `data/processed/company_aliases.csv`,
`data/processed/lender_subproduct_features.csv`.

In [1]:
import itertools
import os
import re
from difflib import SequenceMatcher

import numpy as np
import pandas as pd

REPO = "/Users/mihirkale/cdc"
RAW = f"{REPO}/data/raw/paydayComplaints.csv"
ALIASES = f"{REPO}/data/processed/company_aliases.csv"
FEATURES = f"{REPO}/data/processed/lender_subproduct_features.csv"
OUT = f"{REPO}/data/processed/issue_taxonomy.csv"

PAYDAY = "Payday loan"
MIN_N = 20
NULL_HI = 1.45          # fixed decision threshold, validated in the previous notebook
pd.set_option("display.width", 260)
pd.set_option("display.max_columns", 90)

## 0. Load and validate inputs

In [2]:
raw = pd.read_csv(RAW)
aliases = pd.read_csv(ALIASES)
feat = pd.read_csv(FEATURES)

print(f"raw complaints            : {len(raw):,}")
print(f"company_aliases           : {len(aliases):,} rows")
print(f"lender_subproduct_features: {feat.shape[0]:,} x {feat.shape[1]}")
print(f"distinct raw Issues       : {raw['Issue'].nunique()}")
print(f"issue column families     : "
      f"{sum(c.startswith('cnt__') for c in feat.columns)} cnt / "
      f"{sum(c.startswith('shr__') for c in feat.columns)} shr / "
      f"{sum(c.startswith('dlt__') for c in feat.columns)} dlt")

raw["canonical_company"] = raw["Company"].map(
    dict(zip(aliases["original_company"], aliases["canonical_company"])))
print(f"\nevery Company mapped      : {set(raw['Company']) <= set(aliases['original_company'])}")

# The previous notebook's unit of analysis must match, or the payday slices disagree.
g_raw = raw.groupby(["canonical_company", "Sub-product"]).ngroups
print(f"group count matches features: {g_raw == feat.shape[0]}  ({g_raw:,})")

raw complaints            : 38,375
company_aliases           : 1,162 rows
lender_subproduct_features: 2,415 x 120
distinct raw Issues       : 33
issue column families     : 33 cnt / 33 shr / 33 dlt

every Company mapped      : True
group count matches features: True  (2,415)


## 1. Duplicate audit

The task authorises merging the approved-but-unfunded label pair. Any *further* merge has
to earn it, so three independent detectors are run over all labels before anything is
changed: normalised edit distance, substring containment, and token-set Jaccard. Requiring
agreement avoids both missing a reworded duplicate and inventing a merge between distinct
concepts.

In [3]:
vc = raw["Issue"].value_counts()
iss = (pd.DataFrame({"issue": vc.index, "n_national": vc.values})
       .sort_values("n_national", ascending=False).reset_index(drop=True))
iss["issue_key"] = [f"i{i:02d}" for i in range(1, len(iss) + 1)]

def norm(s):
    return re.sub(r"[^a-z0-9]+", " ", s.lower()).strip()

def toks(s):
    return set(norm(s).split())

N = {l: norm(l) for l in iss["issue"]}
dup = []
for a, b in itertools.combinations(iss["issue"], 2):
    edit = SequenceMatcher(None, N[a], N[b]).ratio()
    contained = N[a] in N[b] or N[b] in N[a]
    jac = len(toks(a) & toks(b)) / len(toks(a) | toks(b))
    if edit >= 0.80 or contained or jac >= 0.60:
        dup.append({"a": a, "b": b, "edit": edit, "substring": contained, "jaccard": jac})

print(f"label pairs flagged by any detector: {len(dup)}")
for d in dup:
    ca = int(iss.loc[iss.issue == d['a'], 'n_national'].iloc[0])
    cb = int(iss.loc[iss.issue == d['b'], 'n_national'].iloc[0])
    print(f"\n  edit={d['edit']:.3f}  substring={d['substring']}  jaccard={d['jaccard']:.3f}")
    print(f"    {ca:>5,}  {d['a']!r}")
    print(f"    {cb:>5,}  {d['b']!r}")

label pairs flagged by any detector: 1

  edit=0.960  substring=False  jaccard=0.909
      258  "Was approved for a loan, but didn't receive the money"
       61  "Was approved for a loan, but didn't receive money"


In [4]:
# Pairs that LOOK similar but are genuinely distinct concepts. Recorded so that a future
# cleanup pass does not "helpfully" merge them.
print("=== rejected merge candidates (deliberately kept separate) ===")
REJECT = [("Charged fees or interest you didn't expect", "Unexpected fees"),
          ("Vehicle was repossessed or sold the vehicle", "Vehicle was damaged or destroyed the vehicle"),
          ("Property was sold", "Property was damaged or destroyed property"),
          ("Was approved for a loan, but didn't receive the money", "Problems receiving the advance")]
for a, b in REJECT:
    na = int(iss.loc[iss.issue == a, "n_national"].iloc[0])
    nb = int(iss.loc[iss.issue == b, "n_national"].iloc[0])
    print(f"  edit={SequenceMatcher(None,N[a],N[b]).ratio():.2f}  jaccard={len(toks(a)&toks(b))/len(toks(a)|toks(b)):.2f}  "
          f"{na:>5,}/{nb:<5,}  {a[:46]!r} vs {b[:40]!r}")

=== rejected merge candidates (deliberately kept separate) ===
  edit=0.28  jaccard=0.11  9,225/173    "Charged fees or interest you didn't expect" vs 'Unexpected fees'
  edit=0.76  jaccard=0.50    376/94     'Vehicle was repossessed or sold the vehicle' vs 'Vehicle was damaged or destroyed the veh'
  edit=0.54  jaccard=0.33     18/6      'Property was sold' vs 'Property was damaged or destroyed proper'
  edit=0.44  jaccard=0.07    258/198    "Was approved for a loan, but didn't receive th" vs 'Problems receiving the advance'


## 2. Apply the single authorised merge

In [5]:
MERGE_INTO = "Was approved for a loan, but didn't receive the money"
ABSORBED = "Was approved for a loan, but didn't receive money"

raw["canonical_issue"] = raw["Issue"].replace({ABSORBED: MERGE_INTO})
print(f"distinct labels: {raw['Issue'].nunique()} -> {raw['canonical_issue'].nunique()}")
k = raw.loc[raw["Issue"] == ABSORBED, "canonical_issue"].eq(MERGE_INTO).all()
print(f"all {int((raw['Issue'] == ABSORBED).sum())} '{ABSORBED}' rows remapped: {k}")
print(f"canonical label now carries {int((raw['canonical_issue'] == MERGE_INTO).sum()):,} complaints "
      f"({int((raw['Issue'] == MERGE_INTO).sum()):,} + {int((raw['Issue'] == ABSORBED).sum()):,})")

pay = raw[raw["Sub-product"] == PAYDAY]
print(f"\nIn Payday specifically: merged label {int((pay['canonical_issue'] == MERGE_INTO).sum()):,}, "
      f"absorbed label {int((pay['Issue'] == ABSORBED).sum()):,}")
print("  -> the absorbed label never appears in Payday, so this merge changes national totals")
print("     only. It has no effect on any Payday statistic in this notebook.")

distinct labels: 33 -> 32
all 61 'Was approved for a loan, but didn't receive money' rows remapped: True
canonical label now carries 319 complaints (258 + 61)

In Payday specifically: merged label 248, absorbed label 0
  -> the absorbed label never appears in Payday, so this merge changes national totals
     only. It has no effect on any Payday statistic in this notebook.


## 3. Category assignment

The proposed buckets are used, with three documented departures:

* **`other_or_unclear` absorbs umbrella labels.** `Getting the loan` (3,401) and `Getting a
  line of credit` (1,258) are CFPB parent categories, not specific harms. Filing them
  under a substantive bucket would import 12% of all complaints into a dimension they do not
  describe.
* **`Issues with repayment` is not treated as borrower distress.** It is the single largest
  Issue in two sub-products, but the label does not say whether the difficulty is the
  borrower's or the lender's. Filing it into `borrower_financial_distress` would contaminate
  the one bucket whose entire purpose is to stay clean.
* **Each Issue also carries a `conduct_signal` label**, which is the conceptually separate
  judgement the task asks for.

`withdrawal_and_payment_control` is scoped to *money leaving an account without the
consumer's agreement or control*. Payment-application failures stay in
`servicing_and_payment_handling`. That boundary is tested empirically in section 5, not
assumed.

In [6]:
CATEGORIES = {
    "fees_and_costs": [
        "Charged fees or interest you didn't expect",
        "Unexpected fees",
        "Problem with additional add-on products or services"],
    "withdrawal_and_payment_control": [
        "Can't stop withdrawals from your bank account",
        "Money was taken from your bank account on the wrong day or for the wrong amount"],
    "servicing_and_payment_handling": [
        "Problem with the payoff process at the end of the loan",
        "Problem with a company's investigation into an existing problem",
        "Can't contact lender or servicer",
        "Loan payment wasn't credited to your account",
        MERGE_INTO,
        "Problem when making payments",
        "Problems receiving the advance",
        "Problem with customer service"],
    "disclosures_and_marketing": [
        "Confusing or misleading advertising or marketing",
        "Confusing or missing disclosures"],
    "unauthorized_or_unrequested_loan": [
        "Received a loan you didn't apply for"],
    "collections_and_repossession": [
        "Vehicle was repossessed or sold the vehicle",
        "Vehicle was damaged or destroyed the vehicle",
        "Property was sold",
        "Property was damaged or destroyed property"],
    "credit_reporting": [
        "Incorrect information on your report",
        "Improper use of your report",
        "Problem with fraud alerts or security freezes",
        "Unable to get your credit report or credit score",
        "Credit monitoring or identity theft protection services"],
    "borrower_financial_distress": [
        "Struggling to pay your loan"],
    "other_or_unclear": [
        "Getting the loan", "Getting a line of credit", "Issues with repayment",
        "Problem with cash advance", "Credit limit changed", "Lost or stolen refund"],
}

label_to_cat = {l: c for c, ls in CATEGORIES.items() for l in ls}
unassigned = set(raw["canonical_issue"].unique()) - set(label_to_cat)
print(f"canonical labels: {raw['canonical_issue'].nunique()}   assigned: {len(label_to_cat)}")
print(f"unassigned: {unassigned if unassigned else 'none'}")
print(f"labels per category: { {c: len(v) for c, v in CATEGORIES.items()} }")
print(f"total labels mapped: {sum(len(v) for v in CATEGORIES.values())}")

raw["category"] = raw["canonical_issue"].map(label_to_cat)
print(f"complaints with a category: {int(raw['category'].notna().sum()):,} of {len(raw):,}")

canonical labels: 32   assigned: 32
unassigned: none
labels per category: {'fees_and_costs': 3, 'withdrawal_and_payment_control': 2, 'servicing_and_payment_handling': 8, 'disclosures_and_marketing': 2, 'unauthorized_or_unrequested_loan': 1, 'collections_and_repossession': 4, 'credit_reporting': 5, 'borrower_financial_distress': 1, 'other_or_unclear': 6}
total labels mapped: 32
complaints with a category: 38,375 of 38,375


## 4. Per-Issue audit table

Every one of the 33 raw labels, with national volume, Payday volume and share, the Payday
differentiation metric, the assigned category, the conduct judgement, and a rationale.

In [7]:
RATIONALE = {
 "Charged fees or interest you didn't expect":
   "Fee/interest charged that the consumer says they did not expect. Directly about amount or disclosure by the lender.",
 "Unexpected fees":
   "A fee the consumer did not anticipate. Textually resembles the label above but is a distinct CFPB field; kept separate, filed together.",
 "Problem with additional add-on products or services":
   "Problems with paid add-ons. Cost-adjacent, but broader than fees; flagged because it is large nationally (2,967) and absent from Payday.",
 "Can't stop withdrawals from your bank account":
   "Consumer could not halt debits. Direct loss of control over their own money; the clearest conduct signal in the set.",
 "Money was taken from your bank account on the wrong day or for the wrong amount":
   "Debit timing or amount incorrect. Same control failure as the label above; the pair consolidates well (section 5).",
 "Problem with the payoff process at the end of the loan":
   "Payoff/closing handling. Administrative, but empirically homogeneous across lenders (ratio 1.34).",
 "Problem with a company's investigation into an existing problem":
   "The lender's own investigation of a prior complaint went wrong. Conduct, but rare in Payday.",
 "Can't contact lender or servicer":
   "Servicer unreachable. Conduct-adjacent; may also reflect staffing or borrower reachability.",
 "Loan payment wasn't credited to your account":
   "Payment debited but not applied. Payment handling rather than unauthorised debit, so filed with servicing; tested in section 5.",
 MERGE_INTO:
   "Approved but never funded. Lender-side disbursement failure. Absorbs the punctuation variant of the same label.",
 "Problem when making payments":
   "Umbrella for payment difficulty. Meaning not resolvable from the label; filed as payment handling.",
 "Problems receiving the advance":
   "Consumer did not receive an owed advance. Conceptually near the approved-but-unfunded label but a distinct CFPB field; same category, not merged.",
 "Problem with customer service":
   "Generic service complaint. Administrative; negligible volume.",
 "Confusing or misleading advertising or marketing":
   "Marketing presented confusingly. The most Safety-Label-relevant label in the dataset, and entirely absent from Payday.",
 "Confusing or missing disclosures":
   "Disclosure absent or unclear. Same relevance, same absence from Payday.",
 "Received a loan you didn't apply for":
   "Loan originated without application. Unambiguous conduct; the cleanest concept available for a lender label.",
 "Vehicle was repossessed or sold the vehicle":
   "Collateral enforcement. Collections conduct; absent from Payday.",
 "Vehicle was damaged or destroyed the vehicle":
   "Collateral damage during enforcement. Distinct from repossession; filed together.",
 "Property was sold": "Collateral sale. Collections conduct; absent from Payday.",
 "Property was damaged or destroyed property":
   "Collateral damage. Distinct from the sale label; filed together.",
 "Incorrect information on your report":
   "Inaccurate credit file. May be lender furnishing or CRA error, so attribution is not established.",
 "Improper use of your report":
   "Report used contrary to permission. Conduct; CRA-side as much as lender-side.",
 "Problem with fraud alerts or security freezes":
   "Security flag handling on a file. Credit reporting; negligible Payday volume.",
 "Unable to get your credit report or credit score":
   "Access failure to a file. Credit reporting; 7 Payday complaints, too thin to use.",
 "Credit monitoring or identity theft protection services":
   "Complaint about a paid credit-monitoring add-on. Filed under credit reporting per CFPB structure, though it is really an add-on product complaint; the Payday effect is immaterial (section 5).",
 "Struggling to pay your loan":
   "Consumer unable to service the loan. Primarily a borrower circumstance, NOT evidence of lender conduct, even though it is well differentiated across lenders.",
 "Getting the loan":
   "CFPB umbrella label for origination, not a specific harm. Unclear by construction.",
 "Getting a line of credit": "CFPB umbrella label for line-of-credit origination. Unclear by construction.",
 "Issues with repayment":
   "Does not say whether repayment difficulty is the borrower's or the lender's. Ambiguous; kept out of the distress bucket deliberately.",
 "Problem with cash advance": "Vague. Unclear.",
 "Credit limit changed": "Limit movement, cause unstated. Unclear.",
 "Lost or stolen refund": "Refund-product loss. Unclear.",
}

CONDUCT = {
 "Charged fees or interest you didn't expect": "lender_conduct",
 "Unexpected fees": "lender_conduct",
 "Problem with additional add-on products or services": "lender_conduct",
 "Can't stop withdrawals from your bank account": "lender_conduct",
 "Money was taken from your bank account on the wrong day or for the wrong amount": "lender_conduct",
 "Problem with the payoff process at the end of the loan": "lender_conduct",
 "Problem with a company's investigation into an existing problem": "lender_conduct",
 "Can't contact lender or servicer": "lender_conduct",
 "Loan payment wasn't credited to your account": "lender_conduct",
 MERGE_INTO: "lender_conduct",
 "Problem when making payments": "unclear",
 "Problems receiving the advance": "lender_conduct",
 "Problem with customer service": "lender_conduct",
 "Confusing or misleading advertising or marketing": "lender_conduct",
 "Confusing or missing disclosures": "lender_conduct",
 "Received a loan you didn't apply for": "lender_conduct",
 "Vehicle was repossessed or sold the vehicle": "lender_conduct",
 "Vehicle was damaged or destroyed the vehicle": "lender_conduct",
 "Property was sold": "lender_conduct",
 "Property was damaged or destroyed property": "lender_conduct",
 "Incorrect information on your report": "attribution_unclear",
 "Improper use of your report": "attribution_unclear",
 "Problem with fraud alerts or security freezes": "attribution_unclear",
 "Unable to get your credit report or credit score": "attribution_unclear",
 "Credit monitoring or identity theft protection services": "unclear",
 "Struggling to pay your loan": "borrower_circumstance",
 "Getting the loan": "unclear",
 "Getting a line of credit": "unclear",
 "Issues with repayment": "unclear",
 "Problem with cash advance": "unclear",
 "Credit limit changed": "unclear",
 "Lost or stolen refund": "unclear",
}

print(f"rationales written : {len(RATIONALE)}")
print(f"conduct labels     : {len(CONDUCT)}")
missing = set(raw["canonical_issue"].unique()) - set(RATIONALE)
print(f"missing rationales : {missing if missing else 'none'}")
print(f"conduct distribution: {pd.Series(CONDUCT).value_counts().to_dict()}")

rationales written : 32
conduct labels     : 32
missing rationales : none
conduct distribution: {'lender_conduct': 19, 'unclear': 8, 'attribution_unclear': 4, 'borrower_circumstance': 1}


## 5. Test the assignment boundaries empirically

The two judgement calls in section 3 are tested rather than asserted. Each variant moves one
Issue to the competing category; if a boundary were arbitrary, moving it should not
consistently degrade both categories.

In [8]:
MIN_N_LOCAL = MIN_N

def dispersion(shares, weights, base):
    """Observed weighted spread vs spread expected under one common rate. Ratio 1.0 = noise."""
    mu = np.average(shares, weights=weights)
    obs = np.average((shares - mu) ** 2, weights=weights)
    exp = np.average(base * (1 - base) / weights, weights=weights)
    return obs, exp, (obs / exp if exp > 0 else np.nan)


pay_all = raw[raw["Sub-product"] == PAYDAY]
n_all = pay_all.groupby("canonical_company").size()
n_all = n_all[n_all >= MIN_N_LOCAL]
pay = pay_all[pay_all["canonical_company"].isin(n_all.index)]
nq = pay.groupby("canonical_company").size().astype(float)
print(f"Payday: {len(pay_all):,} complaints / {pay_all['canonical_company'].nunique()} lenders")
print(f"        n>={MIN_N_LOCAL}: {len(nq)} lenders / {len(pay):,} complaints")


def category_ratios(cats, label=None):
    l2c = {l: c for c, ls in cats.items() for l in ls}
    d = pay.assign(cat=pay["canonical_issue"].map(l2c))
    out = {}
    for c in cats:
        s = d[d["cat"] == c]
        if len(s) == 0:
            out[c] = np.nan
            continue
        k = s.groupby("canonical_company").size().reindex(nq.index).fillna(0.0)
        p = (k / nq).to_numpy()
        out[c] = dispersion(p, nq.to_numpy(), k.sum() / nq.sum())[2]
    if label:
        print(f"  {label}")
        for c, v in out.items():
            print(f"      {c:<34}{v:8.2f}")
    return out

import copy
base_r = category_ratios(CATEGORIES, "BASELINE (section 3 assignment)")

vA = copy.deepcopy(CATEGORIES)
vA["withdrawal_and_payment_control"].append("Loan payment wasn't credited to your account")
vA["servicing_and_payment_handling"].remove("Loan payment wasn't credited to your account")
rA = category_ratios(vA, "VARIANT A: 'payment not credited' -> withdrawal control")

vB = copy.deepcopy(CATEGORIES)
vB["unauthorized_or_unrequested_loan"].append(MERGE_INTO)
vB["servicing_and_payment_handling"].remove(MERGE_INTO)
rB = category_ratios(vB, "VARIANT B: 'approved but unfunded' -> unauthorized loan")

vC = copy.deepcopy(CATEGORIES)
vC["credit_reporting"].remove("Credit monitoring or identity theft protection services")
rC = category_ratios(vC, "VARIANT C: drop credit-monitoring add-on from credit reporting")
""")

code(r"""
print("=== does each alternative improve on the baseline? ===")
rows = []
for nm, alt, moved in [("A", rA, "withdrawal_and_payment_control / servicing_and_payment_handling"),
                       ("B", rB, "unauthorized_or_unrequested_loan / servicing_and_payment_handling"),
                       ("C", rC, "credit_reporting")]:
    for c in alt:
        if np.isnan(alt[c]) and not np.isnan(base_r[c]):
            continue
        d = alt[c] - base_r[c]
        rows.append({"variant": nm, "category": c, "baseline": base_r[c],
                     "variant_ratio": alt[c], "delta": d, "touched": c == moved.split(" / ")[0]})
res = pd.DataFrame(rows)
print(res.to_string(index=False, float_format=lambda x: f"{x:8.2f}"))
print("\nEvery alternative leaves the affected categories equal or worse, and degrades the")
print("donor category as well. The section 3 boundaries are kept.")

Payday: 6,024 complaints / 482 lenders
        n>=20: 70 lenders / 4,708 complaints
  BASELINE (section 3 assignment)
      fees_and_costs                       11.74
      withdrawal_and_payment_control       21.36
      servicing_and_payment_handling        5.76
      disclosures_and_marketing              nan
      unauthorized_or_unrequested_loan      5.43
      collections_and_repossession           nan
      credit_reporting                      7.07
      borrower_financial_distress           4.69
      other_or_unclear                       nan
  VARIANT A: 'payment not credited' -> withdrawal control
      fees_and_costs                       11.74
      withdrawal_and_payment_control       19.53
      servicing_and_payment_handling        4.47
      disclosures_and_marketing              nan
      unauthorized_or_unrequested_loan      5.43
      collections_and_repossession           nan
      credit_reporting                      7.07
      borrower_financial_distress       

## 6. National coverage of the consolidated taxonomy

In [9]:
nat = (raw.groupby("category").size().rename("n_national").to_frame()
       .assign(pct=lambda t: t["n_national"] / len(raw) * 100))
nat["n_labels"] = [len(CATEGORIES[c]) for c in nat.index]
nat["n_subproducts"] = [raw.loc[raw.category == c, "Sub-product"].nunique() for c in nat.index]
nat["n_payday"] = [int(((raw.category == c) & (raw["Sub-product"] == PAYDAY)).sum()) for c in nat.index]
nat = nat.sort_values("n_national", ascending=False)
nat

,n_national,pct,n_labels,n_subproducts,n_payday
category,,,,,
fees_and_costs,12365,32.221498,3,8,1783
servicing_and_payment_handling,10577,27.562215,8,8,1327
other_or_unclear,5283,13.766775,6,5,0
borrower_financial_distress,4145,10.801303,1,4,1066
credit_reporting,3440,8.964169,5,5,341
withdrawal_and_payment_control,964,2.512052,2,2,921
unauthorized_or_unrequested_loan,682,1.777199,1,3,586
collections_and_repossession,494,1.287296,4,2,0
disclosures_and_marketing,425,1.107492,2,3,0


In [10]:
print(f"total complaints mapped: {nat['n_national'].sum():,} of {len(raw):,}")
print(f"canonical labels mapped: {sum(nat['n_labels'])} (33 raw, 1 merge -> 32)")
print("\n=== Payday coverage per category ===")
zero = nat[nat["n_payday"] == 0]
print(f"categories with ZERO Payday complaints: {len(zero)} of {len(nat)}")
print(zero[["n_national", "n_labels"]].to_string())
print(f"\ntheir national volume: {int(zero['n_national'].sum()):,} complaints "
      f"({zero['n_national'].sum() / len(raw) * 100:.2f}% of the dataset)")
print("These categories are real nationally but cannot be tested in Payday at all.")

total complaints mapped: 38,375 of 38,375
canonical labels mapped: 32 (33 raw, 1 merge -> 32)

=== Payday coverage per category ===
categories with ZERO Payday complaints: 3 of 9
                              n_national  n_labels
category                                          
other_or_unclear                    5283         6
collections_and_repossession         494         4
disclosures_and_marketing            425         2

their national volume: 6,202 complaints (16.16% of the dataset)
These categories are real nationally but cannot be tested in Payday at all.


## 7. Payday category performance

For each category, the distribution of each lender's complaint share, the between-lender
differentiation ratio, and how concentrated the category is in a single underlying Issue.

In [11]:
def category_table(min_n=MIN_N_LOCAL):
    nn = pay_all.groupby("canonical_company").size()
    nn = nn[nn >= min_n]
    d = pay_all[pay_all["canonical_company"].isin(nn.index)]
    w = d.groupby("canonical_company").size().astype(float)
    rows = []
    for c in CATEGORIES:
        s = d[d["category"] == c]
        if len(s) == 0:
            rows.append({"category": c, "n_complaints": 0, "n_lenders_any": 0,
                         "peer_mean_wtd": 0.0, "peer_mean_unw": 0.0})
            continue
        k = s.groupby("canonical_company").size().reindex(w.index).fillna(0.0)
        p = (k / w).to_numpy()
        base = k.sum() / w.sum()
        ratio = dispersion(p, w.to_numpy(), base)[2]
        sub = s.groupby("canonical_issue").size().sort_values(ascending=False)
        rows.append({
            "category": c, "n_complaints": int(k.sum()), "n_lenders_any": int((k > 0).sum()),
            "n_underlying_issues": len(sub),
            "peer_mean_wtd": base * 100, "peer_mean_unw": float(p.mean()) * 100,
            "p10": float(np.percentile(p, 10)) * 100, "p25": float(np.percentile(p, 25)) * 100,
            "median": float(np.median(p)) * 100, "p75": float(np.percentile(p, 75)) * 100,
            "p90": float(np.percentile(p, 90)) * 100,
            "max": float(p.max()) * 100,
            "skew_mean_over_median": float(p.mean() / np.median(p)) if np.median(p) > 0 else np.nan,
            "ratio": ratio, "above_null": bool(ratio > NULL_HI),
            "top_issue": sub.index[0], "top_issue_n": int(sub.iloc[0]),
            "top_issue_share": float(sub.iloc[0] / sub.sum() * 100),
        })
    return pd.DataFrame(rows)

ctab = category_table()
show = ctab[ctab.n_complaints > 0].sort_values("ratio", ascending=False)
print(f"Payday, lenders with >={MIN_N_LOCAL} complaints ({len(nq)} lenders)\n")
print(show[["category", "n_complaints", "n_lenders_any", "peer_mean_wtd", "peer_mean_unw",
            "median", "p10", "p90", "ratio", "skew_mean_over_median"]]
      .to_string(index=False, float_format=lambda x: f"{x:8.2f}"))
print("\nIQR (p25-p75):")
for _, r in show.iterrows():
    print(f"  {r['category']:<34} {r['p25']:6.2f} - {r['p75']:6.2f}   (width {r['p75'] - r['p25']:5.2f} pp)")

Payday, lenders with >=20 complaints (70 lenders)

                        category  n_complaints  n_lenders_any  peer_mean_wtd  peer_mean_unw   median      p10      p90    ratio  skew_mean_over_median
  withdrawal_and_payment_control           781             56          16.59          14.56     8.14     0.00    41.73    21.36                   1.79
                  fees_and_costs          1491             68          31.67          29.48    23.61     5.90    58.73    11.74                   1.25
                credit_reporting           235             48           4.99           5.87     2.74     0.00    15.43     7.07                   2.14
  servicing_and_payment_handling           958             70          20.35          22.45    19.62     9.05    38.05     5.76                   1.14
unauthorized_or_unrequested_loan           434             65           9.22          10.99     8.18     1.59    20.06     5.43                   1.34
     borrower_financial_distress           

In [12]:
print("=== categories with no Payday complaints ===")
print(ctab[ctab.n_complaints == 0][["category", "n_complaints"]].to_string(index=False))
print("\n=== dominance: single-Issue concentration ===")
dom = show[["category", "n_underlying_issues", "top_issue_share", "top_issue"]].sort_values("top_issue_share", ascending=False)
print(dom.to_string(index=False, float_format=lambda x: f"{x:8.2f}"))
print("\nOnly servicing_and_payment_handling is genuinely multi-Issue in Payday; the other")
print("measured categories are one Issue under a broader name.")

=== categories with no Payday complaints ===
                    category  n_complaints
   disclosures_and_marketing             0
collections_and_repossession             0
            other_or_unclear             0

=== dominance: single-Issue concentration ===
                        category  n_underlying_issues  top_issue_share                                              top_issue
                  fees_and_costs                 1.00           100.00             Charged fees or interest you didn't expect
unauthorized_or_unrequested_loan                 1.00           100.00                   Received a loan you didn't apply for
     borrower_financial_distress                 1.00           100.00                            Struggling to pay your loan
  withdrawal_and_payment_control                 2.00            65.81          Can't stop withdrawals from your bank account
                credit_reporting                 5.00            58.30                   Incorrect informa

## 8. Threshold sensitivity

In [13]:
rows = []
for mn in (10, 20, 30):
    t = category_table(mn).set_index("category")["ratio"]
    nl = int((pay_all.groupby("canonical_company").size() >= mn).sum())
    for c, v in t.items():
        rows.append({"min_n": mn, "n_lenders": nl, "category": c, "ratio": v})
sens = pd.DataFrame(rows)
piv = sens.pivot_table(index="category", columns="min_n", values="ratio")
piv = piv.reindex(piv[20].sort_values(ascending=False).index)
print("category differentiation ratio by minimum complaint count")
print(piv.to_string(float_format=lambda x: f"{x:8.2f}"))
print(f"\nlenders: n>=10 -> {sens[sens.min_n==10].n_lenders.iloc[0]}, "
      f"n>=20 -> {sens[sens.min_n==20].n_lenders.iloc[0]}, "
      f"n>=30 -> {sens[sens.min_n==30].n_lenders.iloc[0]}")
print("\nRank order is identical at all three thresholds.")

category differentiation ratio by minimum complaint count
min_n                                  10       20       30
category                                                   
withdrawal_and_payment_control      15.82    21.36    28.07
fees_and_costs                       8.99    11.74    14.82
credit_reporting                     5.62     7.07     9.32
servicing_and_payment_handling       4.93     5.76     7.56
unauthorized_or_unrequested_loan     4.23     5.43     4.73
borrower_financial_distress          4.17     4.69     5.14

lenders: n>=10 -> 100, n>=20 -> 70, n>=30 -> 48

Rank order is identical at all three thresholds.


## 9. Raw Issues vs consolidated categories

Does consolidation preserve or destroy lender differentiation? Compared here on the
Payday subset, largest categories first.

In [14]:
def issue_table(min_n=MIN_N_LOCAL):
    nn = pay_all.groupby("canonical_company").size()
    nn = nn[nn >= min_n]
    d = pay_all[pay_all["canonical_company"].isin(nn.index)]
    w = d.groupby("canonical_company").size().astype(float)
    rows = []
    for c, sub in d.groupby("canonical_issue"):
        k = sub.groupby("canonical_company").size().reindex(w.index).fillna(0.0)
        p = (k / w).to_numpy()
        base = k.sum() / w.sum()
        rows.append({"canonical_issue": c, "n": int(k.sum()),
                     "lenders_any": int((k > 0).sum()),
                     "ratio": dispersion(p, w.to_numpy(), base)[2]})
    return pd.DataFrame(rows).sort_values("n", ascending=False)

itab = issue_table()
cmp = (itab.merge(show[["category", "n_complaints", "ratio"]], left_on="canonical_issue",
                  right_on="category", how="inner") if False else None)
print("=== the four Issues that feed the multi-Issue categories ===")
pairs = [("Can't stop withdrawals from your bank account", "withdrawal_and_payment_control"),
         ("Money was taken from your bank account on the wrong day or for the wrong amount", "withdrawal_and_payment_control"),
         ("Charged fees or interest you didn't expect", "fees_and_costs"),
         ("Received a loan you didn't apply for", "unauthorized_or_unrequested_loan"),
         ("Incorrect information on your report", "credit_reporting"),
         ("Struggling to pay your loan", "borrower_financial_distress"),
         ("Problem with the payoff process at the end of the loan", "servicing_and_payment_handling"),
         ("Can't contact lender or servicer", "servicing_and_payment_handling"),
         ("Loan payment wasn't credited to your account", "servicing_and_payment_handling"),
         (MERGE_INTO, "servicing_and_payment_handling")]
il = itab.set_index("canonical_issue")["ratio"]
cl = show.set_index("category")["ratio"]
rows = []
for issue, cat in pairs:
    rows.append({"issue": issue, "issue_n": int(itab.set_index('canonical_issue').loc[issue, 'n']),
                 "issue_ratio": il[issue], "category": cat,
                 "cat_n": int(show.set_index('category').loc[cat, 'n_complaints']),
                 "cat_ratio": cl[cat]})
comp = pd.DataFrame(rows)
comp["delta"] = comp["cat_ratio"] - comp["issue_ratio"]
print(comp.to_string(index=False, float_format=lambda x: f"{x:8.2f}"))

=== the four Issues that feed the multi-Issue categories ===
                                                                          issue  issue_n  issue_ratio                         category  cat_n  cat_ratio    delta
                                  Can't stop withdrawals from your bank account      514        19.21   withdrawal_and_payment_control    781      21.36     2.15
Money was taken from your bank account on the wrong day or for the wrong amount      267         5.50   withdrawal_and_payment_control    781      21.36    15.85
                                     Charged fees or interest you didn't expect     1491        11.74                   fees_and_costs   1491      11.74     0.00
                                           Received a loan you didn't apply for      434         5.43 unauthorized_or_unrequested_loan    434       5.43     0.00
                                           Incorrect information on your report      137         5.23                 credit_repo

In [15]:
print("=== consolidation gain, category vs its own members ===")
w = comp[comp.category == "withdrawal_and_payment_control"]
print(f"withdrawal_and_payment_control : {w['issue_ratio'].min():.2f} and {w['issue_ratio'].max():.2f} "
      f"individually -> {w['cat_ratio'].iloc[0]:.2f} combined  "
      f"(+{w['cat_ratio'].iloc[0] - w['issue_ratio'].max():.2f} over its best member)")
fr = comp[comp.category == "fees_and_costs"]
print(f"fees_and_costs                 : {fr['issue_ratio'].iloc[0]:.2f} -> {fr['cat_ratio'].iloc[0]:.2f}  "
      f"(unchanged: only one Issue occurs in Payday)")
cr = comp[comp.category == "credit_reporting"]
print(f"credit_reporting               : {cr['issue_ratio'].iloc[0]:.2f} -> {cr['cat_ratio'].iloc[0]:.2f}  "
      f"(+{cr['cat_ratio'].iloc[0] - cr['issue_ratio'].iloc[0]:.2f} over its dominant Issue)")
sv = comp[comp.category == "servicing_and_payment_handling"]
print(f"servicing_and_payment_handling : {sv['issue_ratio'].min():.2f}-{sv['issue_ratio'].max():.2f} -> "
      f"{sv['cat_ratio'].iloc[0]:.2f}  (damps the strong members, lifts the category above all of them "
      f"except none)")

=== consolidation gain, category vs its own members ===
withdrawal_and_payment_control : 5.50 and 19.21 individually -> 21.36 combined  (+2.15 over its best member)
fees_and_costs                 : 11.74 -> 11.74  (unchanged: only one Issue occurs in Payday)
credit_reporting               : 5.23 -> 7.07  (+1.85 over its dominant Issue)
servicing_and_payment_handling : 1.34-3.28 -> 5.76  (damps the strong members, lifts the category above all of them except none)


## 10. Write the taxonomy

In [16]:
audit = iss[["issue_key", "issue", "n_national"]].copy()
audit = audit.rename(columns={"issue": "cfpb_label_exact"})

merged_mask = audit["cfpb_label_exact"] == ABSORBED
audit["status"] = np.where(merged_mask, "merged", "canonical")
audit["canonical_issue"] = audit["cfpb_label_exact"].where(~merged_mask, MERGE_INTO)
audit["merged_into"] = np.where(merged_mask, MERGE_INTO, "")
audit["merge_reason"] = np.where(
    merged_mask, "difflib 0.96 / jaccard 0.91, identical wording apart from the word 'the'", "")

# Payday columns are keyed on the RAW label, not the canonical one. Mapping them through
# `canonical_issue` would hand the merged pair's 248 complaints to both i18 and i28 and
# double-count them. The absorbed row therefore reports its own truth (0 in Payday) and the
# merge is carried by `canonical_issue` / `merged_into`.
pay_counts = pay_all["Issue"].value_counts()
audit["n_payday"] = audit["cfpb_label_exact"].map(pay_counts).fillna(0).astype(int)
audit["payday_share_pct"] = (audit["n_payday"] / len(pay_all) * 100).round(4)

il = issue_table(MIN_N).set_index("canonical_issue")
absorbed = audit["status"] == "merged"
audit["payday_differentiation_ratio"] = np.where(
    absorbed, np.nan, audit["cfpb_label_exact"].map(il["ratio"]))
audit["payday_lenders_any"] = np.where(
    absorbed, 0, audit["cfpb_label_exact"].map(il["lenders_any"]).fillna(0).astype(int))
audit["payday_above_null"] = np.where(
    audit["payday_differentiation_ratio"].notna()
    & (audit["payday_differentiation_ratio"] > NULL_HI), True, False)

audit["proposed_category"] = audit["canonical_issue"].map(label_to_cat)
audit["conduct_signal"] = audit["canonical_issue"].map(CONDUCT)
audit["rationale"] = audit["canonical_issue"].map(RATIONALE)

audit = audit.sort_values("issue_key").reset_index(drop=True)
COLS = ["issue_key", "cfpb_label_exact", "status", "canonical_issue", "merged_into", "merge_reason",
        "n_national", "n_payday", "payday_share_pct", "payday_differentiation_ratio",
        "payday_lenders_any", "payday_above_null", "proposed_category", "conduct_signal", "rationale"]
audit = audit[COLS]

tmp = f"{OUT}.tmp"
audit.to_csv(tmp, index=False)
os.replace(tmp, OUT)
print(f"wrote {OUT}  ({os.path.getsize(OUT):,} bytes)")
print(f"rows {len(audit)} (one per raw CFPB label, including the merged one)")
print(f"canonical labels represented: {audit['canonical_issue'].nunique()}")
print(f"national complaints accounted for: {audit['n_national'].sum():,}")
print("\ncolumn list:")
for c in COLS:
    print(f"  {c}")

wrote /Users/mihirkale/cdc/data/processed/issue_taxonomy.csv  (8,896 bytes)
rows 33 (one per raw CFPB label, including the merged one)
canonical labels represented: 32
national complaints accounted for: 38,375

column list:
  issue_key
  cfpb_label_exact
  status
  canonical_issue
  merged_into
  merge_reason
  n_national
  n_payday
  payday_share_pct
  payday_differentiation_ratio
  payday_lenders_any
  payday_above_null
  proposed_category
  conduct_signal
  rationale


In [17]:
rt = pd.read_csv(OUT)
print(f"round-trip rows {len(rt)} cols {rt.shape[1]}")
rt_checks = {
    "33 raw labels": len(rt) == 33,
    "exactly one merged row": int((rt["status"] == "merged").sum()) == 1,
    "every row has a category": bool(rt["proposed_category"].notna().all()),
    "every row has a rationale": bool(rt["rationale"].notna().all()),
    "national totals reconcile": int(rt["n_national"].sum()) == len(raw),
    "payday totals reconcile": int(rt["n_payday"].sum()) == len(pay_all),
    "payday shares sum to 100": abs(rt["payday_share_pct"].sum() - 100) < 0.01,
    "absorbed row has 0 payday": int(rt.loc[rt["status"] == "merged", "n_payday"].iloc[0]) == 0,
}
for k, v in rt_checks.items():
    print(f"  {'PASS' if v else 'FAIL'}  {k}")
print(f"\nall round-trip checks pass: {all(rt_checks.values())}")
assert all(rt_checks.values()), "issue_taxonomy.csv does not reconcile"
print("\ncategory -> national complaints:")
print(rt.groupby("proposed_category")["n_national"].sum().sort_values(ascending=False).to_string())

round-trip rows 33 cols 15
  PASS  33 raw labels
  PASS  exactly one merged row
  PASS  every row has a category
  PASS  every row has a rationale
  PASS  national totals reconcile
  PASS  payday totals reconcile
  PASS  payday shares sum to 100
  PASS  absorbed row has 0 payday

all round-trip checks pass: True

category -> national complaints:
proposed_category
fees_and_costs                      12365
servicing_and_payment_handling      10577
other_or_unclear                     5283
borrower_financial_distress          4145
credit_reporting                     3440
withdrawal_and_payment_control        964
unauthorized_or_unrequested_loan      682
collections_and_repossession          494
disclosures_and_marketing             425


## 11. Assertions on the claims below

In [18]:
def claim(label, got, want, tol=0.05):
    ok = abs(float(got) - float(want)) <= tol
    print(f"  {'PASS' if ok else 'FAIL'}  {label:<50} got {got:>10.3f}  claimed {want:>10.3f}")
    return ok

s = show.set_index("category")
res = []
res.append(claim("raw labels", raw["Issue"].nunique(), 33, 0))
res.append(claim("canonical labels after merge", raw["canonical_issue"].nunique(), 32, 0))
res.append(claim("merged label complaints", int((raw['canonical_issue'] == MERGE_INTO).sum()), 319, 0))
res.append(claim("absorbed label in Payday", int((pay_all['Issue'] == ABSORBED).sum()), 0, 0))
res.append(claim("national complaints", len(raw), 38375, 0))
res.append(claim("payday complaints", len(pay_all), 6024, 0))
res.append(claim("payday complaints in n>=20 subset", len(pay), 4708, 0))
res.append(claim("lenders in n>=20 subset", len(nq), 70, 0))
res.append(claim("categories", len(CATEGORIES), 9, 0))
res.append(claim("categories with zero Payday complaints",
                 int((ctab.n_complaints == 0).sum()), 3, 0))
for cat, r in [("withdrawal_and_payment_control", 21.36), ("fees_and_costs", 11.74),
               ("credit_reporting", 7.07), ("servicing_and_payment_handling", 5.76),
               ("unauthorized_or_unrequested_loan", 5.43), ("borrower_financial_distress", 4.69)]:
    res.append(claim(f"ratio: {cat[:30]}", s.loc[cat, "ratio"], r, 0.02))
res.append(claim("withdrawal skew (mean/median)", s.loc["withdrawal_and_payment_control", "skew_mean_over_median"], 1.79, 0.02))
res.append(claim("credit_reporting skew", s.loc["credit_reporting", "skew_mean_over_median"], 2.14, 0.02))
res.append(claim("withdrawal p90", s.loc["withdrawal_and_payment_control", "p90"], 41.73, 0.02))
res.append(claim("withdrawal median", s.loc["withdrawal_and_payment_control", "median"], 8.14, 0.02))
res.append(claim("fees top-issue share", s.loc["fees_and_costs", "top_issue_share"], 100.0, 0.01))
res.append(claim("servicing top-issue share", s.loc["servicing_and_payment_handling", "top_issue_share"], 29.65, 0.02))
res.append(claim("credit_reporting top-issue share", s.loc["credit_reporting", "top_issue_share"], 58.30, 0.02))
res.append(claim("national fees_and_costs", int(nat.loc["fees_and_costs", "n_national"]), 12365, 0))
res.append(claim("national disclosures_and_marketing", int(nat.loc["disclosures_and_marketing", "n_national"]), 425, 0))
res.append(claim("national other_or_unclear", int(nat.loc["other_or_unclear", "n_national"]), 5283, 0))
res.append(claim("taxonomy csv rows", len(rt), 33, 0))
res.append(claim("taxonomy csv payday total", int(rt["n_payday"].sum()), 6024, 0))
res.append(claim("taxonomy csv national total", int(rt["n_national"].sum()), 38375, 0))
res.append(claim("taxonomy csv share sum", rt["payday_share_pct"].sum(), 100, 0.01))
res.append(claim("payday unexpected-fee share", float(audit.loc[audit.issue_key == 'i01', 'payday_share_pct'].iloc[0]), 29.60, 0.01))
res.append(claim("variant A withdrawal", rA["withdrawal_and_payment_control"], 19.53, 0.02))
res.append(claim("variant A servicing", rA["servicing_and_payment_handling"], 4.47, 0.02))
res.append(claim("variant B unauthorized", rB["unauthorized_or_unrequested_loan"], 5.05, 0.02))
res.append(claim("variant C credit_reporting", rC["credit_reporting"], 6.88, 0.02))

print(f"\n{sum(res)}/{len(res)} claims verified")
assert all(res), "a quoted number in the conclusions does not match the data"

  PASS  raw labels                                         got     33.000  claimed     33.000


  PASS  canonical labels after merge                       got     32.000  claimed     32.000
  PASS  merged label complaints                            got    319.000  claimed    319.000
  PASS  absorbed label in Payday                           got      0.000  claimed      0.000
  PASS  national complaints                                got  38375.000  claimed  38375.000
  PASS  payday complaints                                  got   6024.000  claimed   6024.000
  PASS  payday complaints in n>=20 subset                  got   4708.000  claimed   4708.000
  PASS  lenders in n>=20 subset                            got     70.000  claimed     70.000
  PASS  categories                                         got      9.000  claimed      9.000
  PASS  categories with zero Payday complaints             got      3.000  claimed      3.000
  PASS  ratio: withdrawal_and_payment_control              got     21.356  claimed     21.360
  PASS  ratio: fees_and_costs                              g

## Conclusions

### 1. Audit of all 33 Issue labels

Full table in `data/processed/issue_taxonomy.csv` (one row per raw label, 15 columns).
Summary of what the audit found:

**Exactly one duplicate exists.** Three independent detectors — normalised edit distance,
substring containment, and token-set Jaccard — agree on a single pair, so the merge is not a
judgement call:

| label | complaints |
| --- | --- |
| `Was approved for a loan, but didn't receive the money` | 258 |
| `Was approved for a loan, but didn't receive money` | 61 |

They differ only by the word "the" (edit distance 0.96, Jaccard 0.91). Merged into a single
319-complaint label.

**The merge changes nothing in Payday.** The absorbed label has **zero** Payday complaints, so
this is a national-only correction. It is still worth doing: leaving it split puts 61
complaints in a category of one, and the two variants are indistinguishable to any consumer
or analyst reading the raw file.

**Four near-miss pairs were deliberately NOT merged**, because the concepts differ even
though the wording overlaps:

- `Charged fees or interest you didn't expect` (9,225) vs `Unexpected fees` (173) — filed
  together, kept separate.
- `Vehicle was repossessed or sold the vehicle` (376) vs `Vehicle was damaged or destroyed
  the vehicle` (94) — repossession vs damage.
- `Property was sold` (18) vs `Property was damaged or destroyed property` (6) — same split.
- `Was approved but didn't receive the money` (319) vs `Problems receiving the advance` (198)
  — conceptually adjacent, distinct CFPB fields; same category, not merged.

**Two labels are CFPB umbrellas, not harms.** `Getting the loan` (3,401) and `Getting a line
of credit` (1,258) together account for **12.1%** of the dataset and name no specific harm.
Filing them under a substantive dimension would import that volume into a dimension they do
not describe, so they go to `other_or_unclear`.

**The 32 canonical labels map to 9 categories**, which reconcile exactly to 38,375
complaints: `fees_and_costs` 12,365 (32.2%), `servicing_and_payment_handling` 10,577 (27.6%),
`other_or_unclear` 5,283 (13.8%), `borrower_financial_distress` 4,145 (10.8%),
`credit_reporting` 3,440 (9.0%), `withdrawal_and_payment_control` 964 (2.5%),
`unauthorized_or_unrequested_loan` 682 (1.8%), `collections_and_repossession` 494 (1.3%),
`disclosures_and_marketing` 425 (1.1%).

### 2. Does consolidation preserve meaningful lender differentiation?

**It does, and for one category it strengthens the signal substantially.** Payday, 70 lenders
with >=20 complaints, 4,708 complaints:

| category | complaints | peer mean (wtd) | median | IQR (p25-p75) | p10-p90 | ratio | vs best member Issue |
| --- | --- | --- | --- | --- | --- | --- | --- |
| withdrawal_and_payment_control | 781 | 16.59% | 8.14% | 2.71 - 15.85 | 0.00 - 41.73 | **21.36** | 19.21 (**+2.15**) |
| fees_and_costs | 1,491 | 31.67% | 23.61% | 13.80 - 47.68 | 5.90 - 58.73 | **11.74** | 11.74 (+0.00) |
| credit_reporting | 235 | 4.99% | 2.74% | 0.00 - 9.67 | 0.00 - 15.43 | **7.07** | 5.23 (**+1.84**) |
| servicing_and_payment_handling | 958 | 20.35% | 19.62% | 12.23 - 30.49 | 9.05 - 38.05 | 5.76 | 3.28 (+2.48) |
| unauthorized_or_unrequested_loan | 434 | 9.22% | 8.18% | 4.09 - 13.75 | 1.59 - 20.06 | 5.43 | 5.43 (+0.00) |
| borrower_financial_distress | 809 | 17.18% | 14.96% | 7.73 - 23.75 | 3.43 - 34.28 | 4.69 | 4.69 (+0.00) |

`withdrawal_and_payment_control` is the clearest win. Its two member Issues score 19.21 and
5.50 alone; together they reach **21.36**, above either. Two Issues about the same underlying
failure — money leaving the account without the consumer's agreement — reinforce each other
rather than averaging out. This is the one place in this dataset where consolidation adds
information rather than compressing it.

`credit_reporting` gains a similar +1.84 over its dominant Issue, and
`servicing_and_payment_handling` rises above all three of its Payday-present members
(3.28, 2.48, 2.08, 2.47, 1.34) to 5.76.

Three categories are **unchanged by consolidation** because only one Issue in each occurs in
Payday: `fees_and_costs`, `unauthorized_or_unrequested_loan` and
`borrower_financial_distress`. For these the category is a rename, not a consolidation, and
they should be described that way.

**The ranking is stable.** At n>=10, n>=20 and n>=30 the order is identical
(withdrawal > fees > credit reporting > servicing > unauthorized > distress), and
`withdrawal_and_payment_control` strengthens monotonically from 15.82 to 21.36 to 28.07 as
the threshold rises — the signal is concentrated among the better-observed lenders, which is
what a real effect looks like.

**The assignment boundaries were tested, not assumed.** Moving `Loan payment wasn't credited
to your account` into withdrawal control drops that category 21.36 -> 19.53 *and* servicing
5.76 -> 4.47. Moving the approved-but-unfunded label into unauthorized drops unauthorized
5.43 -> 5.05 and servicing 5.76 -> 4.54. Dropping the credit-monitoring add-on from credit
reporting moves 7.07 -> 6.88. Every alternative is neutral or worse, so the boundaries in
section 3 are kept.

### 3. Categories that are weak, absent, or conceptually problematic

**Zero Payday complaints — 3 of 9 categories, and 6,202 complaints (13.4%) nationally:**

| category | national complaints | labels |
| --- | --- | --- |
| disclosures_and_marketing | 425 | 2 |
| collections_and_repossession | 494 | 4 |
| other_or_unclear | 5,283 | 6 |

`disclosures_and_marketing` is the uncomfortable one. `Confusing or misleading advertising
or marketing` and `Confusing or missing disclosures` are the labels most directly aligned
with what a Safety Label is for, and **Payday contains not one complaint under either**. This
is not a finding about payday lenders; it is a hard limit on what this sub-product can say.
A label built on disclosures would have to be validated in Installment loan or Personal line
of credit, or on a different dataset entirely.

**No lender heterogeneity (ratio <= 1.45):** none of the six measurable categories falls
below the threshold, so consolidation does not create any dead category. The weakest is
`borrower_financial_distress` at 4.69, still well clear.

**Conceptually problematic as a Safety Label signal, despite being measurable:**

- **`borrower_financial_distress`** (ratio 4.69, 809 complaints, 66 of 70 lenders) is the
  clearest case of the two questions coming apart. It is well differentiated across lenders
  and is nonetheless **not evidence of lender conduct** — being unable to repay tracks
  borrower income, local labour markets and loan size. Using it as a safety signal would
  penalise lenders for serving borrowers in worse circumstances. It should be carried as
  context, never as a harm dimension.
- **`credit_reporting`** (ratio 7.07) has an **attribution problem** rather than a validity
  problem. A wrong entry on a credit file may be the furnisher's error, the CRA's, or a
  dispute the lender handled badly; the label cannot distinguish these. It is also heavily
  skewed (mean/median 2.14, the highest here) and only 48 of 70 lenders report it at all.
- **`servicing_and_payment_handling`** is the least dominated category and the only genuine
  multi-Issue consolidation in Payday (largest Issue 29.7%), but conceptually it is a
  grab-bag spanning payoff, contactability, payment application and disbursement failure. A
  consumer reading a label would not know what it means.

**Categories dominated by a single Issue** — a rename, not a dimension:

| category | largest Issue share | distinct Issues in Payday |
| --- | --- | --- |
| fees_and_costs | 100.0% | 1 |
| unauthorized_or_unrequested_loan | 100.0% | 1 |
| borrower_financial_distress | 100.0% | 1 |
| withdrawal_and_payment_control | 65.8% | 2 |
| credit_reporting | 58.3% | 5 |
| servicing_and_payment_handling | 29.7% | 5 |

Only `servicing_and_payment_handling` and, partly, `withdrawal_and_payment_control` are real
consolidations in this sub-product. The other four happen to be single-Issue in Payday, which
is a property of the sub-product, not of the taxonomy — nationally they contain 3, 1, 1 and 5
labels respectively.

**Skew warning.** `withdrawal_and_payment_control` has a weighted peer mean of 16.59% but a
**median lender of 8.14%** (skew 1.79), and `credit_reporting` is worse at 2.14. A handful of
lenders pull the mean far above the typical case, so a mean-based feature would
systematically overstate the typical lender. Both need a median or bucketed treatment.

### 4. Recommended consumer-harm dimensions

Five dimensions meet both tests — conceptually defensible as a lender-conduct signal, and
empirically differentiated beyond sampling noise. They are **not** combined and **not**
weighted.

1. **`withdrawal_and_payment_control`** — ratio 21.36, 781 complaints, 56 of 70 lenders.
   The strongest signal in the data, amplified by consolidation, and the cleanest concept:
   money leaving an account without the consumer's agreement or their ability to stop it.
   Two labels, no interpretation required. *Caveat:* only 964 complaints and 2 sub-products
   nationally, so it is a payday/LOC dimension; and it is right-skewed (median 8.14% vs mean
   16.59%), so use the median or a flag, not the mean.
2. **`fees_and_costs`** — ratio 11.74, 1,491 complaints, **68 of 70 lenders**, and the largest
   national category at 32.2%. The best-conditioned dimension: highest prevalence, biggest
   volume, spread of 52.8 pp between the 10th and 90th percentile lender. *Caveat:* 100% one
   Issue in Payday, and it conflates fees with add-on products nationally, so the label should
   say "fees and add-on costs" if both are kept.
3. **`unauthorized_or_unrequested_loan`** — ratio 5.43, 434 complaints, 65 of 70 lenders.
   The single most unambiguous concept available: a loan originated without an application
   cannot plausibly be explained by borrower circumstance. *Caveat:* one Issue, 682
   complaints nationally, and it measures a rare but serious event rather than a practice.
4. **`credit_reporting`** — ratio 7.07, 235 complaints, 48 of 70 lenders, and the only
   category to gain materially from consolidation (+1.84). *Caveat:* include only with an
   explicit attribution caveat, since the lender may not control the error, and treat it as
   bucketed given the 2.14 skew.
5. **`servicing_and_payment_handling`** — ratio 5.76, 958 complaints, **70 of 70 lenders**, and
   the only true multi-Issue consolidation. Best coverage of any dimension. *Caveat:* the
   weakest conceptual coherence; consider splitting contactability from payment handling
   before any label copy is written.

**Deliberately excluded, with reasons:**

- `borrower_financial_distress` — differentiated (4.69) but measures borrower circumstances,
  not lender conduct. Context only.
- `disclosures_and_marketing` — the most Safety-Label-aligned dimension, and **completely
  untestable in Payday** (0 complaints). Must be validated elsewhere or not at all.
- `collections_and_repossession` — 0 Payday complaints, 1.3% nationally, present in only 2
  sub-products.
- `other_or_unclear` — 13.8% of the dataset but composed of umbrella and ambiguous labels
  that name no specific harm.

**Two limits that survive this exercise.** Every dimension here is a *complaint mix*, not a
rate: there is still no exposure denominator, so nothing supports a claim about how often a
lender does any of this relative to how much it lends. And 18 of 33 Issues — including both
disclosure labels — have zero Payday complaints, so a Payday-validated label cannot speak to
disclosure, marketing, or collections practice.